# 🏗️ Notebook 1: Amazon Lambda (Serverless) — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A FaaS (Function-as-a-Service) platform. Users upload code + handler; we run it on demand in response to events (HTTP, queue, timer). They pay only for execution time.

Core tension: **fast starts** vs **isolation** vs **cost**.

## Requirements

### Functional
- Upload code; set runtime + handler + memory.
- Invoke via HTTP or event source.
- Scale to 1000s of concurrent invocations.
- Per-invocation metrics & logs.

### Non-functional
- Warm start < 10ms, cold start < 1s.
- Hard isolation between tenants.
- Pay-per-ms billing.

## Back-of-envelope

- 10M functions. Active at any moment: 1% = 100k.
- Avg container memory: 256 MB → 25 TB RAM across fleet.
- Cold start rate: ~1% of invocations. Target: shrink this.

## High-level architecture

```
  [Event source] ──► Front End ──► Placement Svc
                         │                │
                         │                ▼
                         │           Warm pool (per function)
                         │                │  miss
                         │                ▼
                         │           Cold init (Firecracker microVM)
                         │                │
                         │                ▼
                         │           Running container
                         ▼
                     Metrics + Logs
```

- Each tenant runs in a **microVM** for hard isolation.
- Warm containers held for N minutes to avoid cold starts on repeat invocations.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.